# 📘 Capítulo 4 – Avaliação e Validação Temporal
**Autor:** Rodrigo Santana Ferreira  
**Disciplina:** Séries Temporais  

---
Este notebook contém os scripts Python apresentados no Capítulo 4, organizados por seção conforme o material da aula.

## 🔧 Instalação de Dependências

In [ ]:
# Instalação das bibliotecas necessárias
!pip install pandas numpy scikit-learn matplotlib

## 📦 Passo 1 – Preparação dos Dados e Criação de Features
Carregamento do dataset AirPassengers e criação das features temporais básicas (seção HANDS ON).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Carregar dataset clássico
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url, parse_dates=['Month'], index_col='Month')
df.columns = ['Passengers']
df = df.asfreq('MS')

# Criar features temporais básicas
df['lag_1'] = df['Passengers'].shift(1)
df['lag_12'] = df['Passengers'].shift(12)
df['diff_1'] = df['Passengers'].diff(1)
df['rolling_mean_3'] = df['Passengers'].rolling(3).mean()
df['month'] = df.index.month
df['sin_month'] = np.sin(2 * np.pi * df['month'] / 12)
df['cos_month'] = np.cos(2 * np.pi * df['month'] / 12)

df = df.dropna()
X = df.drop('Passengers', axis=1)
y = df['Passengers']

print(f"Shape final: {X.shape}")

## ❌ Passo 2 – Validação Errada: O que NUNCA Fazer
Demonstração do problema de data leakage com validação aleatória em séries temporais.

In [ ]:
from sklearn.model_selection import train_test_split

# Validação aleatória (errada para séries temporais)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

print(f"MAE (validação incorreta): {mean_absolute_error(y_test, pred):.2f}")
print("→ Este resultado é ilusório! O modelo viu o futuro.")

## ✅ Passo 3 – Validação Correta com TimeSeriesSplit
Avaliação respeitando a ordem temporal — sem vazamento de dados futuros.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
mae_scores, rmse_scores, mape_scores = [], [], []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = GradientBoostingRegressor(random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mape = np.mean(np.abs((y_test - pred) / y_test)) * 100

    mae_scores.append(mae)
    rmse_scores.append(rmse)
    mape_scores.append(mape)

    print(f"Fold {fold}: MAE = {mae:.2f} | RMSE = {rmse:.2f} | MAPE = {mape:.2f}%")

print(f"\nMédia TimeSeriesSplit → MAE: {np.mean(mae_scores):.2f} | RMSE: {np.mean(rmse_scores):.2f} | MAPE: {np.mean(mape_scores):.2f}%")

## 🔄 Passo 4 – Rolling Forecast (Walk-Forward Validation)
Simulação do uso real em produção: re-treino a cada janela de previsão.

In [ ]:
def rolling_forecast_evaluation(X, y, test_size=12, n_windows=5):
    mae_list = []
    for i in range(n_windows):
        train_end = len(X) - (n_windows - i) * test_size
        X_train = X.iloc[:train_end]
        y_train = y.iloc[:train_end]
        X_test = X.iloc[train_end:train_end + test_size]
        y_test = y.iloc[train_end:train_end + test_size]

        model = GradientBoostingRegressor(random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        mae_list.append(mean_absolute_error(y_test, pred))
    return np.mean(mae_list)

print(f"MAE médio (Rolling Forecast): {rolling_forecast_evaluation(X, y):.2f}")

## 📏 Passo 5 – Comparação com Modelo Naïve (Baseline)
O modelo naïve é o benchmark mínimo: qualquer modelo sério deve superá-lo.

In [ ]:
# Previsão naïve: valor do período anterior
naive_pred = y.shift(1).dropna()
y_naive = y.iloc[1:]

print(f"MAE Naïve: {mean_absolute_error(y_naive, naive_pred):.2f}")